# Data collection: web scraping

Isse Abdallah Mohamoud. IBM Applied Data Science Capstone historical snapshots. Run from the spacex-capstone folder.

In [1]:
from pathlib import Path
import json, pandas as pd
from IPython.display import display
R = Path('.')
import requests, io
from bs4 import BeautifulSoup
status = json.loads((R/'collection_status.json').read_text())
display(pd.DataFrame([status]))

,wiki_http_status,wiki_url,note
0,403,https://en.wikipedia.org/w/index.php?title=Lis...,Historical endpoint rejected requests during p...


The historical Wikipedia revision returned HTTP 403. The parsing function below is complete, but no fresh historical extraction is claimed. Downstream analyses use the separately supplied course snapshots.

In [2]:
def scrape_launch_tables(html):
    soup = BeautifulSoup(html, 'html.parser')
    tables = [t for t in soup.find_all('table')
              if 'Payload' in t.get_text() and 'Launch' in t.get_text()]
    frames = []
    for table in tables:
        frames.extend(pd.read_html(io.StringIO(str(table))))
    for frame in frames:
        for col in frame.select_dtypes('object'):
            frame[col] = frame[col].astype(str).str.replace(r'\[.*?\]', '', regex=True).str.strip()
    return soup.title.get_text() if soup.title else None, frames

print('Parser defined. Historical retrieval status:', status['wiki_http_status'])


Parser defined. Historical retrieval status: 403


To retry with network access, call requests.get(status["wiki_url"]), require a successful status, then pass response.text to scrape_launch_tables. collect_data.py implements this guarded request.